## Model Evaluation: Walk-Forward Cross Validation

Before comparing SARIMA forecasts against live World Cup data, the modeel's accuracy will be validated using walk-forward cross-validation. This ensures the baseline forecasts are reliable enough to meaningfully measure World Cup deviations.

**Methodology:**
- Split each city's time series into 4 folds
- Each fold trains on all data before the test window (no data leakage)
- Test window is 6 months per fold
- Metrics: RMSE and MAPE averaged across all folds and cities

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import TimeSeriesSplit 
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.statespace.sarimax import SARIMAX
from pmdarima import auto_arima
import sys
sys.path.append("..")
from constants import target_cities, host_cities, nonhost_cities


In [8]:
# Reading in data
df = pd.read_csv('../data/processed/worldcup_city_seasonal_spending.csv', index_col='Date', parse_dates = True)
print(df.head())

            spend_all  spend_acf     cityname stateabbrev  city_pop2019  \
Date                                                                      
2022-01-31    -0.1300    -0.1580  Los Angeles          CA      10039107   
2022-01-31     0.1810     0.0257      Chicago          IL       5150233   
2022-01-31    -0.0201    -0.0351       Dallas          TX       2635516   
2022-01-31     0.0388     0.0112       Austin          TX       1273954   
2022-01-31    -0.0339    -0.0717    Charlotte          NC       1110356   

            season host_status  
Date                            
2022-01-31  Winter        Host  
2022-01-31  Winter    Non-Host  
2022-01-31  Winter        Host  
2022-01-31  Winter    Non-Host  
2022-01-31  Winter    Non-Host  


In [ ]:
# Testing on Atlanta data to ensure pipeline works end-to-end and for debugging purposes

atl_df = df[df['cityname'] == 'Atlanta']
atl_spending = atl_df.spend_acf

# Splitting data into 3 folds

time_split = TimeSeriesSplit(n_splits=2, test_size=12)  # 2 splits with a test size of 12 months each

rmse_scores = [] # store rmse scores to calculate average at the end of loop
mape_scores = [] # store mape scores to calculate average at the end of loop

# Loop through the splits and train/test the model
for train_idx, test_idx in time_split.split(atl_spending):

    atl_train, atl_test = atl_spending.iloc[train_idx], atl_spending.iloc[test_idx]
    
    # Fitting SARIMA model
    #auto_model = auto_arima(atl_train, seasonal=True, m=12, D=1, max_D=1, error_action='ignore', suppress_warnings=True)
    auto_model = auto_arima(atl_train, seasonal=True, m=12, seasonal_test='ch', error_action='ignore', suppress_warnings=True)
    sarima_model = SARIMAX(atl_train, order=auto_model.order, seasonal_order=auto_model.seasonal_order)
    model_fit = sarima_model.fit(disp=False) 
    
    # Generate forecasts from training data
    forecast = model_fit.get_forecast(steps=len(atl_test))
    forecast_mean = forecast.predicted_mean
    forecast_mean.name = "Predicted Forecast" 

    # Computing RMSE (Root Mean Squared Error)
    mse = mean_squared_error(atl_test, forecast_mean)
    rmse = np.sqrt(mse) 
    print(f"RMSE ({atl_test.index[0].strftime('%b %Y')} - {atl_test.index[-1].strftime('%b %Y')}): {rmse:.4f}")
    rmse_scores.append(rmse)

    # MAPE (Mean Absolute Percentage Error)
    mape = np.mean(np.abs((atl_test.values - forecast_mean.values) / atl_test.values)) * 100
    print(f"MAPE ({atl_test.index[0].strftime('%b %Y')} - {atl_test.index[-1].strftime('%b %Y')}): {mape:.4f}")
    mape_scores.append(mape)
    print()

    # MASE (Mean Absolute Scaled Error)
    naive_errors = np.abs(np.diff(atl_train.values))
    mae_naive = np.mean(naive_errors)
    mae_model = np.mean(np.abs(atl_test.values - forecast_mean.values))
    mase = mae_model / mae_naive
    print(f"MASE ({atl_test.index[0].strftime('%b %Y')} - {atl_test.index[-1].strftime('%b %Y')}): {mase:.4f}")
    mase_scores.append(mase)

RMSE (Jun 2024 - May 2025): 0.0676
MAPE (Jun 2024 - May 2025): 40.4347

MASE (Jun 2024 - May 2025): 1.3130


NameError: name 'mase_scores' is not defined

RMSE is used over MSE because it returns error in the original units of the data (percent change), making it directly interpretable. A lower RMSE indicates the model's forecasts are closer to actual spending values.

The Jun 2024–May 2025 window shows a higher MAPE (36.4%) than Jun 2025–May 2026 (29.6%), despite RMSE also improving (0.0629 → 0.0424). This suggests the model's forecasts became more accurate in the more recent window. However, a MAPE in the 30–36% range is relatively high, which is common when forecasting a series with values near zero (percent-change data can produce large percentage errors even from small absolute misses, since dividing by a small actual value inflates the percentage). RMSE is therefore the more reliable metric here for judging absolute accuracy, while MAPE should be read with that caveat in mind rather than taken at face value. 

Add to last notebook for validating model in business context.
These values show that the model had a strong accuracy when predicting Atlanta's spending prices meaning any observed spending above the forecasted baseline can be attributed to World Cup-driven demand rather than normal seasonal variation.

In [10]:
# Conducting model evaluation across all cities

# Initializing lists to track RMSE and MAPE scores
rmse_scores = []
mape_scores = []

for city in target_cities:
    city_df = df[df['cityname'] == city]
    city_spending = city_df.spend_acf

    # Splitting the data into train/test 

    time_split = TimeSeriesSplit(n_splits=2, test_size=12)

    for train_idx, test_idx in time_split.split(city_spending):
        city_train, city_test = city_spending.iloc[train_idx], city_spending.iloc[test_idx]

        # Fitting SARIMA model
        #auto_model = auto_arima(city_train, seasonal=True, m=12, error_action='ignore', suppress_warnings='True')
        auto_model = auto_arima(city_train, seasonal=True, m=12, seasonal_test='ch', error_action='ignore', suppress_warnings=True)
        print(f"{city} order: {auto_model.order}, seasonal_order: {auto_model.seasonal_order}")
        sarima_model = SARIMAX(city_train, order=auto_model.order, seasonal_order=auto_model.seasonal_order, trend='c' if auto_model.with_intercept else None)
        #sarima_model = SARIMAX(city_train, order=auto_model.order, seasonal_order=auto_model.seasonal_order)
        model_fit = sarima_model.fit(disp=False)

        # Forecast generatoin 
        forecast = model_fit.get_forecast(steps=len(city_test))
        forecast_mean = forecast.predicted_mean
        print(forecast_mean.values)

        print(f"{city}")
        # Computing RMSE (Root Mean Squared Error)
        mse = mean_squared_error(city_test, forecast_mean)
        rmse = np.sqrt(mse) 
        print(f"RMSE ({city_test.index[0].strftime('%b %Y')} - {city_test.index[-1].strftime('%b %Y')}): {rmse:.4f}")
        rmse_scores.append(rmse)

        # MAPE (Mean Absolute Percentage Error)
        mape = np.mean(np.abs((city_test.values - forecast_mean.values) / city_test.values)) * 100
        print(f"MAPE ({city_test.index[0].strftime('%b %Y')} - {city_test.index[-1].strftime('%b %Y')}): {mape:.4f}")
        mape_scores.append(mape)
        print()
    

Los Angeles order: (0, 0, 2), seasonal_order: (1, 0, 1, 12)
[-0.1300638  -0.141648   -0.1817938  -0.15599821 -0.16328165 -0.17882007
 -0.14243973 -0.14299954 -0.13610291 -0.13239397 -0.15640191 -0.1448782 ]
Los Angeles
RMSE (Jun 2024 - May 2025): 0.0195
MAPE (Jun 2024 - May 2025): 11.8919

Los Angeles order: (0, 0, 1), seasonal_order: (1, 0, 0, 12)
[-0.15949521 -0.15093219 -0.18335974 -0.16279593 -0.16042318 -0.16675051
 -0.15014128 -0.11375916 -0.12799564 -0.12245923 -0.1359048  -0.13511388]
Los Angeles
RMSE (Jun 2025 - May 2026): 0.0165
MAPE (Jun 2025 - May 2026): 10.4431

Denver order: (1, 0, 0), seasonal_order: (1, 0, 0, 12)
[ 0.01312367  0.04684437  0.01708315  0.04889417  0.03954871 -0.00072948
 -0.0023817   0.0044443   0.0438919   0.03312647  0.03457443  0.02014913]
Denver
RMSE (Jun 2024 - May 2025): 0.0245
MAPE (Jun 2024 - May 2025): 195.3605

Denver order: (0, 0, 0), seasonal_order: (2, 0, 0, 12)
[ 0.01331748  0.03591269  0.02029804  0.03503846  0.03926496 -0.00267679
  0.0059

In [12]:
for city, fold_num in [('Denver', 1), ('Denver', 2), ('Charlotte', 1), ('Charlotte', 2), ('Boston', 1), ('Dallas', 1), ('Dallas', 2)]:
    city_spending = df[df['cityname'] == city].spend_acf
    splits = list(TimeSeriesSplit(n_splits=2, test_size=12).split(city_spending))
    test_idx = splits[fold_num - 1][1]
    test_vals = city_spending.iloc[test_idx].values
    print(f"{city} fold {fold_num}: min={test_vals.min():.4f}, max={test_vals.max():.4f}, closest to zero={test_vals[np.argmin(np.abs(test_vals))]:.5f}")

Denver fold 1: min=-0.0295, max=0.0540, closest to zero=-0.00412
Denver fold 2: min=-0.0418, max=0.0688, closest to zero=-0.00041
Charlotte fold 1: min=-0.0384, max=0.0440, closest to zero=-0.00395
Charlotte fold 2: min=-0.0782, max=0.0757, closest to zero=-0.00326
Boston fold 1: min=-0.0151, max=0.2610, closest to zero=-0.00773
Dallas fold 1: min=-0.0267, max=0.1120, closest to zero=0.00351
Dallas fold 2: min=-0.0340, max=0.0761, closest to zero=0.00373
